# 07 · CUDA 커널 개념 & CuPy 고수준 커널

> **CuPy 2일 집중 코스 — Day 2 / 단원 5 (CuPy 고수준 커널 + 커널 개념, 1.0H)**

Day 2는 **사용자 정의 커널**입니다. 먼저 **CUDA 커널의 핵심 개념**을 충실히 잡고,
그 개념들이 **07→08→09→10→11** 노트북에서 각각 어떻게 구현되는지 지도를 그립니다.
그다음 가장 쉬운 시작인 CuPy의 `ElementwiseKernel`·`ReductionKernel`을 다룹니다.

### 왜 이 노트북이 중요한가

Day 1(`00`-`06`)은 CuPy가 **이미 만들어둔** 커널(`cp.sin`, `cp.sum`, cuBLAS 행렬곱 등)을 어떻게
효율적으로 *사용*할지 — 포팅, 벤치마킹, 메모리 관리, 스트림 — 를 다뤘습니다. Day 2(`07`-`13`)는
그 커널의 **내부를 직접 설계**하는 단계로 넘어갑니다. `07`은 Day 2 전체의 "개념 사전"에 해당합니다 —
여기서 정의하는 용어(스레드/블록/그리드, warp, coalescing, atomic, occupancy)는 이후 노트북에서
계속 다시 등장하며, 여기서 개념이 흐릿하면 `08`의 coalescing 벤치마크나 `09`의 atomic 히스토그램,
`11`의 RawKernel 최적화가 *왜* 그렇게 동작하는지 이해하기 어려워집니다.

> 비유: `07`은 운전을 배우기 전 "엔진이 어떻게 동력을 만드는가"를 배우는 시간입니다. 이후 노트북은
> 실제로 운전(구현)하는 것이고, `07`이 없으면 감(느낌)만으로 운전하게 됩니다.

### 이 노트북의 구조 — Part A / Part B

- **Part A(1~8절)** 는 순수 개념입니다. 코드가 거의 없는 이유는, 이 개념들이 CUDA C/Numba
  수준에서 벌어지는 일이고 CuPy의 고수준 API를 쓰는 한 대부분 **자동으로 처리**되기 때문입니다.
  그래도 "자동으로 처리된다"는 사실 자체를 알아야, 나중에 08~11에서 그 자동화를 걷어내고
  직접 제어할 때 무엇을 하고 있는지 알 수 있습니다.
- **Part B(9~11절)** 에서 그 개념 위에 놓인 CuPy의 첫 커스텀 커널 API — `ElementwiseKernel`,
  `ReductionKernel` — 를 실제로 작성해 봅니다. 이들은 인덱싱·메모리 계층·동기화를 CuPy가
  대신 처리해주는, Day 2에서 **가장 쉬운** 커널 작성 방법입니다.

## 학습 목표
- CUDA **실행 모델**(커널·thread/block/grid·SIMT/warp)을 설명한다.
- **메모리 계층**·**coalescing**·**동기화/atomic**·**occupancy** 개념을 이해한다.
- 이 개념들이 각 노트북(07~11)에서 어떻게 구현되는지 연결한다.
- `ElementwiseKernel`·`ReductionKernel`로 첫 커스텀 커널을 작성한다.

## 목차
### A. CUDA 커널 개념
1. [커널과 실행 모델](#1) · 2. [스레드 계층과 인덱싱](#2) · 3. [SIMT와 warp](#3)
4. [메모리 계층](#4) · 5. [메모리 접근(coalescing)](#5) · 6. [동기화와 atomic](#6) · 7. [occupancy](#7)
8. [개념 → 노트북 매핑](#8)
### B. CuPy 고수준 커널
9. [ElementwiseKernel](#9) · 10. [ReductionKernel](#10) · 11. [@cupy.fuse 관계](#11) · 12. [체크포인트](#12)

In [ ]:
import os, sys, time, math
import numpy as np
import cupy as cp
from numba import cuda
from course_utils import print_env, bench, gpu_ms, cpu_ms, print_bench, compare
print_env()

<a id="1"></a>
## 1. 커널과 실행 모델

**커널(kernel)** 은 *GPU에서 수천 개의 스레드로 동시에 실행되는 함수*입니다.
CPU(host)가 커널을 **런치(launch)** 하면 GPU가 지정한 수의 스레드를 만들어 같은 코드를 각자 다른 데이터에 적용합니다.
- 호스트가 `kernel[그리드, 블록](인자)` 로 실행 구성을 정해 런치
- 각 스레드는 **자신의 인덱스**로 처리할 데이터를 고름 (SPMD)
- 런치는 **비동기** (Day 1의 비동기와 동일)

> Day 1까지는 CuPy가 커널을 자동 생성했습니다. Day 2는 그 커널을 **직접** 들여다보고 작성합니다.

### 조금 더 구체적으로

- **런치 문법**: CUDA C에서는 `kernel<<<grid, block>>>(args)`, Numba CUDA(`08`-`09`)에서는
  `kernel[grid, block](args)`, RawKernel(`11`)도 CUDA C 문법을 그대로 씁니다. 세 방식 모두
  "그리드 크기·블록 크기를 정하고 런치한다"는 같은 개념을 표현만 다르게 할 뿐입니다. 반면 이번
  노트북 9-11절의 `ElementwiseKernel`/`ReductionKernel`은 이 문법 자체를 **사용자에게 숨깁니다** —
  CuPy가 배열의 크기를 보고 그리드/블록을 알아서 정해 런치합니다.
- **SPMD (Single Program, Multiple Data)**: 모든 스레드가 **완전히 같은 커널 코드**를 실행하지만,
  스레드마다 다른 인덱스를 읽어 다른 데이터에 적용합니다("같은 프로그램, 다른 데이터"). 이는
  하드웨어가 실제로 명령어를 실행하는 방식인 **SIMT**(3절)와는 다른 층위의 개념입니다 — SPMD는
  "프로그래밍 모델"(사용자가 어떻게 코드를 작성하는가), SIMT는 "실행 방식"(하드웨어가 그 코드를
  어떻게 물리적으로 굴리는가)입니다.
- **비동기 런치**: `06_streams_async`에서 다룬 비동기 실행 모델이 커널 런치에도 그대로 적용됩니다.
  `kernel[grid, block](args)` 호출은 GPU의 실행 큐(스트림)에 "이 작업을 실행하라"는 명령만
  등록하고, 파이썬은 즉시 다음 줄로 넘어갑니다. `00`에서 본 "동기화 없이 측정하면 착시가 생긴다"는
  교훈이 Day 2의 커널 런치에도 동일하게 적용되므로, 이후 노트북에서도 `cuda.synchronize()`나
  `bench`로 시간을 잽니다.
- **반환값이 없는 이유**: 커널 함수는 보통 `void`(값을 반환하지 않음)입니다. 결과는 **인자로
  전달된 배열에 직접 씀**으로써 돌려줍니다(위 예시의 `C[i] = ...`처럼) — 함수가 값을 계산해서
  `return`하는 일반적인 파이썬 함수와는 다른 사고방식입니다.

<a id="2"></a>
## 2. 스레드 계층과 인덱싱

| 계층 | 메모리 공간 | A100 기준 용량 | 특징 및 제어 방식 |
|------|------|-----------|----------|
| **thread** | Register | 256 KB (SM 1개당) | 엄청나게 큰 레지스터 파일. 덕분에 수많은 스레드가 Context Switch 비용 없이 대기 가능 |
| **block** | Shared Memory & L1 Cache | 192 KB (SM 1개당) (물리적으로 동일한 공간)  | 하나의 192KB SRAM 칩을 쪼개서(Partition) 사용. 비율 조절 가능 |
| **grid** | L2 Cache | 40 MB (GPU 전체)  | 모든 SM이 공유하며, Global Memory의 병목을 크게 줄여줌 |
| **grid** | Global Memory | 40 GB 또는 80 GB | 가장 용량이 크고 느림 (초당 최대 2TB 대역폭). CPU와 통신하는 메인 메모리 |

```

- NVIDIA A100을 기준으로, 하나의 SM(Streaming Multiprocessor)의 최대 블록과 스레드 수
  * 최대 스레드 수 (Max Threads per SM): 2048개 (즉, 64개의 Warp)
  * 최대 블록 수 (Max Blocks per SM): 32개.   
  * (참고) 블록당 최대 스레드 수: 1024개  
  * (참고) 블록당 스레드 수를 32로 설정시, 최대 블록수가 32이므로 최대 32*32=1024개 까지만 활용 가능(50%만 활용). 

[ GPU Device / Global Memory (VRAM) ] ──────────▶ Software: Grid
  │
  ├── [ L2 Cache ] (모든 SM이 공유하는 자동 캐시)
  │     │
  │     ├── [ SM 0 / L1 Cache & Shared Mem ] ───▶ Software: Block 0
  │     │     │
  │     │     ├── [ CUDA Core / Register ] ─────▶ Software: Thread 0
  │     │     ├── [ CUDA Core / Register ] ─────▶ Software: Thread 1
  │     │     └── ...
  │     │
  │     └── [ SM 1 / L1 Cache & Shared Mem ] ───▶ Software: Block 1
  │           │
  │           ├── [ CUDA Core / Register ] ─────▶ Software: Thread 0
  │           └── ...
  │
  └── ...
  ```

**전역 인덱스**(1D): `global = blockIdx.x*blockDim.x + threadIdx.x`
- `threadIdx.x` 블록 내 위치 · `blockIdx.x` 블록 번호 · `blockDim.x` 블록당 스레드 수
- Numba는 `cuda.grid(1)` 이 위 식을 대신 계산(08). 
   * GPU 스레드의 전역 번호(인덱스)를 구하는 복잡한 수학 공식을 매번 쓸 필요 없이, Numba에서는 cuda.grid(1) 한 줄로 편하게 가져올 수 있음 
   * 뒤의 (1)은 1차원 배열을 의미하며, 2차원/3차원 배열은 cuda.grid(2)

### 조금 더 구체적으로

**숫자로 감을 잡기**: 아래 3.1/3.2절 코드 셀에서 쓰는 `N = 30_000_000`, `threads_per_block = 256`을
그대로 대입하면 `blocks = ceil(N / 256) ≈ 117,188`개의 블록이 필요합니다. 즉 이 한 번의 런치로
`117,188 × 256 ≈ 3천만` 개의 스레드가 논리적으로 동시에 존재합니다 — 물리적인 코어 수(수천 개)보다
훨씬 많죠. 이 "논리적 스레드 수 ≫ 물리적 코어 수" 구조가 바로 3절의 latency hiding을 가능하게
하는 전제입니다.

**하드웨어 한계치**: `blockDim`(블록당 스레드 수)은 보통 **최대 1024개**로 제한됩니다(아키텍처마다
다를 수 있으나 이 숫자가 오랫동안 표준입니다). `gridDim`(그리드당 블록 수)은 x차원 기준 20억 개
이상까지 허용되어 사실상 거의 모든 문제 크기를 한 번의 런치로 덮을 수 있습니다. 반면 y·z차원은
훨씬 작은 한도(보통 65,535)를 가지므로, 아주 큰 2D/3D 문제는 이 한도를 고려해 블록/그리드 크기를
설계해야 합니다.

**2D 인덱싱**: 이미지·행렬처럼 2차원 데이터는 x·y 두 축을 각각 인덱싱합니다.
```
col = blockIdx.x * blockDim.x + threadIdx.x   # 열
row = blockIdx.y * blockDim.y + threadIdx.y   # 행
```
Numba에서는 `row, col = cuda.grid(2)`가 이 두 식을 한 번에 계산해줍니다. 이 패턴은 `09`의 2D
히스토그램류 예제나 이미지 처리에서 그대로 재사용됩니다.

> 인덱싱은 "내가 이 문제 공간 어디에 있는가"를 스레드 스스로 계산하는 유일한 방법입니다 — GPU
> 커널에는 파이썬의 `for i in range(n)` 같은 순차 루프가 없고, 대신 "모든 스레드가 동시에 자기
> 몫만 계산"하는 방식으로 같은 일을 합니다.

<a id="3"></a>
## 3. SIMT와 warp

하드웨어는 스레드를 **warp(32개)** 단위로 묶어 **SIMT**(한 명령을 여러 스레드 동시)로 실행합니다.
- warp의 32 스레드는 **같은 명령을 32명의 스레드가 동시에 똑같은 연산(lockstep)*우ㄹ수행
- warp 내 **분기(divergence)** → 경로별 직렬화로 성능 저하
- 많은 warp로 메모리 지연을 **겹쳐 숨김**(latency hiding)

> 시사점: **균일 연산·연속 접근**이 유리(08 coalescing, 분기 최소화).

### 조금 더 구체적으로

- **warp = 32는 하드웨어 상수**: NVIDIA GPU는 세대(Pascal·Ampere·Hopper 등)에 상관없이 warp
  크기가 32로 고정되어 있습니다. SM(Streaming Multiprocessor, GPU 내부의 "연산 코어 묶음")에는
  여러 개의 warp 스케줄러가 있어, 한 클럭마다 "지금 당장 실행 가능한" warp를 골라 명령을 발행합니다.
- GPU는 CPU와 비교할 수 없을 정도로 거대한 용량의 레지스터를 가지고 있습니다. 따라서 여러 Warp(쓰레드 묶음)가 각자의 데이터를 레지스터에 계속 올려둔 채로 대기할 수 있습니다. 스케줄러가 다. Warp로 실행을 전환할 때, 기존 데이터를 메모리에 저장하고 새 데이터를 불러올(Save & Restore) 필요 없이 그냥 다른 레지스터 영역을 읽기만 하면 되므로 Context Switch 비용이 거의 0에 가깝습니다.
- **분기(divergence) 예시**: `if (threadIdx.x % 2 == 0) { A 코드 } else { B 코드 }` 같은 코드를
  만나면, 한 warp 안의 32개 스레드 중 절반은 조건이 참, 절반은 거짓입니다. 하드웨어는 두 경로를
  **동시에** 실행할 수 없으므로, 먼저 A 경로를 실행하며 조건이 거짓인 스레드는 결과를 버리도록
  마스킹하고, 그다음 B 경로를 실행하며 반대로 마스킹합니다 — 결과적으로 **두 경로의 실행 시간을
  합친 만큼**(최악의 경우 약 2배) 걸립니다. 그래서 "균일한 연산"이 유리하다는 시사점이 나옵니다.
- **latency hiding 숫자 감각**: 전역 메모리(4절)에 접근하는 데는 대략 **수백 클럭 사이클**(흔히
  400~800 사이클 수준으로 이야기됩니다)이 걸립니다. GPU 클럭이 대략 1~1.5 GHz라면 이는 수백
  나노초에 해당하는, 연산 자체(보통 1클럭 내외)에 비해 압도적으로 긴 시간입니다. 이 긴 대기 시간을
  그냥 흘려보내지 않고, 스케줄러가 **다른 준비된 warp로 즉시 전환**해 계산을 계속 시키는 것이
  latency hiding입니다. 바로 다음 두 셀(3.1/3.2)에서 이를 실제 코드로 측정합니다.

### 3.1 CASE 1 (Warp 부족):

GPU 연산 코어(SM)에 일할 수 있는 Warp가 몇 개 없습니다. 메모리에서 A[i], B[i] 데이터를 가져오는 수백 클럭 동안 대신 일해줄 다른 Warp가 없어서 GPU 코어들이 멍하니 대기(Stall)하게 됩니다. 그 결과 데이터 처리 효율(Throughput)이 아주 좋지 않음

SM 입장에서 보면, warp 하나가 `A[i]`/`B[i]`를 전역 메모리에서 읽어오라고 요청을 낸 순간부터
데이터가 도착할 때까지(앞 절에서 이야기한 수백 클럭) 그 warp는 다음 명령을 실행할 수 없는
**stall(정지)** 상태입니다. 이때 교대로 돌릴 다른 warp가 SM에 거의 없다면, 스케줄러는 달리
할 일이 없어 SM의 연산 유닛이 그대로 놀게 됩니다 — 코드 셀의 `blocks_few = 125`(총 스레드
32,000개, warp 약 1,000개)가 바로 이런 "일감이 부족한" 상황을 인위적으로 재현합니다.

### 3.2 CASE 2 (Warp 풍족 - Latency Hiding):

GPU에 수십만 개의 Warp가 빽빽하게 줄을 서 있습니다. Warp 1번이 메모리를 읽어오느라 대기 상태에 빠지는 순간, 스케줄러가 0.1나노초만에 Warp 2번, Warp 3번으로 체인지하여 계산을 계속 돌립니다. 그 사이에 Warp 1번의 데이터가도착하므로, GPU 코어는 단 1클럭도 쉬지 않고 가동되어 전체 처리량이 상승.

코드 셀의 `blocks_many`는 `N = 30_000_000`(3천만) 원소 전체를 처리하도록 스레드를 던져,
warp 수를 약 93만 개로 늘립니다. SM 하나가 동시에 상주시킬 수 있는 warp 수에는 물리적 한계(7절
occupancy에서 다룸)가 있지만, 그 한계까지만 채워도 "대기 중인 warp가 항상 몇 개는 있는" 상태가
되어 메모리 지연이 연산 시간 뒤에 완전히 **가려집니다**. 아래 코드 셀은 CASE 1과 CASE 2의
처리량(M-elements/sec)을 직접 측정해 이 차이를 수치로 보여줍니다 — 실행해서 실제로 몇 배
차이가 나는지 확인해보세요.

In [ ]:
# 메모리 접근(지연)을 일으키는 아주 단순한 커널
@cuda.jit
def memory_heavy_kernel(A, B, C):
    i = cuda.grid(1)
    if i < A.size:
        # [지연 발생] VRAM(Global Memory)에서 데이터를 읽어옴 (수백 클럭 대기)
        a = A[i]
        b = B[i]
        
        # 간단한 수학 연산
        C[i] = a * b + 1.0

# 데이터 준비 (3천만 개 원소)
N = 30_000_000
A = cuda.to_device(np.ones(N, dtype=np.float32))
B = cuda.to_device(np.ones(N, dtype=np.float32))
C = cuda.device_array(N, dtype=np.float32)

threads_per_block = 256 # 블록당 8개의 Warp (256 / 32 = 8 Warps)

# -------------------------------------------------------------------
#  [CASE 1] Warp가 부족할 때 (Latency Hiding 실패)
# 일부러 데이터의 아주 작은 일부(32,000개 = Warp 10개 분량)만 처리하도록 스레드를 적게 던짐
# -------------------------------------------------------------------
blocks_few = 125  # 총 스레드: 125 * 256 = 32,000개 (Warp 1,000개 미만)

start = time.perf_counter()
memory_heavy_kernel[blocks_few, threads_per_block](A, B, C)
cuda.synchronize()
time_few = time.perf_counter() - start

# -------------------------------------------------------------------
#  [CASE 2] Warp가 아주 많을 때 (Latency Hiding 성공)
# 전체 데이터 3천만 개를 모두 처리하도록 대량의 스레드/Warp를 GPU에 던짐
# -------------------------------------------------------------------
blocks_many = (N + threads_per_block - 1) // threads_per_block # 총 스레드: 3,000만 개 (Warp 약 93만 개!)

start = time.perf_counter()
memory_heavy_kernel[blocks_many, threads_per_block](A, B, C)
cuda.synchronize()
time_many = time.perf_counter() - start

# -------------------------------------------------------------------
# 결과 비교 (초당 처리하는 데이터 개수: Throughput 계산)
# -------------------------------------------------------------------
throughput_few = (blocks_few * threads_per_block) / time_few / 1e6
throughput_many = N / time_many / 1e6

print(f"CASE 1 (Warp 부족) Throughput : {throughput_few:.2f} M-elements/sec")
print(f"CASE 2 (Warp 풍족) Throughput : {throughput_many:.2f} M-elements/sec")
print(f"=> Latency Hiding을 통해 초당 처리량이 약 {throughput_many / throughput_few:.1f}배 증가!")

<a id="4"></a>
## 4. 메모리 계층

| 메모리 | 범위 | 속도 | 용도 |
|--------|------|------|------|
| 레지스터 | 스레드 | 가장 빠름 | 지역 변수 |
| 공유(shared) | **블록** | 매우 빠름(온칩) | 블록 내 협력·재사용 |
| 전역(global) | 전체 | 느림(off-chip) | 입출력 배열 |
| 상수/텍스처 | 전체(읽기) | 캐시됨 | 읽기 전용 |

> 전략: **전역 접근↓**, 자주 쓰는 데이터를 **공유/레지스터**에 올려 재사용(09에서 구현).

### 조금 더 구체적으로 — 대략적인 속도·용량 감각

- **레지스터**: 스레드 전용, 접근 지연이 사실상 **1클럭**. 다만 총량이 제한적(SM 전체에 걸쳐
  보통 수만 개의 32비트 레지스터)이라, 스레드 하나가 레지스터를 많이 쓰면 그만큼 SM에 동시에
  올릴 수 있는 스레드 수가 줄어듭니다 — 이 트레이드오프가 7절 occupancy로 이어집니다.
- **공유 메모리(shared)**: 온칩(on-chip) SRAM으로 지연이 대략 **수십 클럭** 수준 — 전역 메모리
  대비 한 자릿수 이상 빠릅니다. 다만 SM당 총 용량이 보통 수십~200KB대로 제한되어 있어, 블록이
  요청하는 공유 메모리 크기가 커질수록 SM에 동시에 상주할 수 있는 블록 수가 줄어듭니다. `09`에서
  히스토그램을 공유 메모리에 부분 집계한 뒤 마지막에 전역으로 합치는 패턴으로 이 자원을 직접 씁니다.
- **전역 메모리(global)**: `00`에서 본 VRAM 전체(수 GB-수십 GB)가 여기 해당하며, 접근 지연은
  3절에서 본 대로 대략 **수백 클럭**입니다. 대역폭 자체는 크지만(수백 GB/s-수 TB/s), 지연이 크기
  때문에 "얼마나 자주, 어떤 패턴으로" 접근하느냐가 실효 성능을 좌우합니다 — 이것이 5절 coalescing의
  핵심 동기입니다.
- **상수(constant) 메모리**: 물리적으로는 전역 메모리의 일부이지만 전용 캐시를 거치며, 보통
  용량이 작습니다(전형적으로 64KB 수준). 모든 스레드가 **같은 주소**를 동시에 읽을 때
  (broadcast) 특히 유리합니다 — 예를 들어 커널 전체에서 공유하는 상수 계수·룩업 테이블에 적합합니다.

> **비유**: 레지스터는 "지금 손에 든 것", 공유 메모리는 "같은 팀 캐비닛"(팀원끼리는 빠르게
> 꺼내 쓰지만 다른 팀은 못 봄), 전역 메모리는 "회사 창고"(누구나 접근 가능하지만 왕복이 오래
> 걸림), 상수 메모리는 "게시판 공지"(모두가 같은 값을 자주 볼 때 특히 효율적)에 가깝습니다.

<a id="5"></a>
## 5. 메모리 접근 — coalescing

전역 메모리는 **연속 주소를 한 트랜잭션**으로 읽을 때 가장 효율적입니다.
- **Coalesced(연속)**: warp 인접 스레드가 인접 주소 → 적은 트랜잭션 ✅
- **Strided/blocked(흩어짐)**: 멀리 떨어진 주소 → 트랜잭션 폭증 ⚠️

같은 일이라도 **접근 패턴**만으로 수 배 차이(08에서 직접 측정).

### 조금 더 구체적으로

메모리 컨트롤러는 개별 바이트가 아니라 **정해진 크기의 캐시라인 단위**(흔히 32바이트 단위, 경우에
따라 더 큰 단위로 뭉쳐서)로 데이터를 실어 나릅니다. 이 사실이 coalescing의 근거입니다.

- **Coalesced 예**: warp의 32개 스레드가 `float32`(4바이트) 배열을 `x[i]` 형태로 인접하게
  읽으면, 32개 스레드가 요구하는 주소 범위는 `32 × 4 = 128`바이트의 연속 구간입니다. 하드웨어는
  이를 최소한의 트랜잭션(캐시라인 몇 개)으로 한 번에 실어올 수 있습니다 — **필요한 만큼만** 정확히
  가져오는 셈입니다.
- **Strided 예**: 만약 각 스레드가 `x[i * stride]`처럼 캐시라인 크기보다 멀리 떨어진 주소를
  읽는다면, 32개 스레드의 요청이 32개의 **서로 다른** 캐시라인에 걸쳐 흩어집니다. 하드웨어는
  결국 각 스레드마다(또는 몇 개씩 묶어) 별도의 트랜잭션을 발행해야 하므로, 트랜잭션 수가 최대
  32배까지 늘어날 수 있고 유효 대역폭은 그만큼 떨어집니다 — 실제로 요청한 데이터양은 같은데,
  훨씬 많은 바이트를 "낭비"해서 실어오는 셈입니다.
- 이 차이는 연산량이 전혀 바뀌지 않아도(같은 덧셈, 같은 곱셈) **접근 패턴만 바꾸는 것만으로**
  수 배의 처리량 차이를 만듭니다. `08_numba_copy`에서 stride 파라미터를 바꿔가며 이 대역폭
  차이를 직접 측정합니다.

> 코딩 습관: 배열을 순회할 때 "warp 안의 스레드 i가 인접한 주소를 읽는가?"를 항상 먼저 확인하는
> 습관을 들이면, 이후 커널 최적화의 8~9할은 이미 절반쯤 끝난 것입니다.

<a id="6"></a>
## 6. 동기화와 atomic

여러 스레드가 같은 메모리를 동시에 수정하면 **데이터 레이스**로 결과가 깨집니다.
- **`atomic`**: 읽기-수정-쓰기를 나눌 수 없는 한 번으로 (`cuda.atomic.add`)
- **`syncthreads()`**: 블록 내 스레드를 한 지점에서 동기화(공유메모리 사용 전후)

> 09 히스토그램: 레이스 → atomic → 공유메모리+atomic 으로 단계 수정.

### 조금 더 구체적으로

- **데이터 레이스가 왜 생기는가**: `hist[bin] += 1` 같은 코드는 실제로 (1) `hist[bin]`을 읽고,
  (2) 1을 더하고, (3) 다시 `hist[bin]`에 쓰는 **세 단계**로 이뤄집니다. 서로 다른 두 스레드가
  같은 `bin`에 동시에 이 세 단계를 밟으면, 두 스레드 모두 "증가 전" 값을 읽어버려서 실제로는
  2가 증가해야 할 값이 1만 증가하는 **lost update**가 벌어집니다. 이는 스레드 수가 많을수록,
  같은 주소로 몰릴수록(충돌·contention) 더 자주 일어납니다.
- **atomic의 역할**: `cuda.atomic.add(hist, bin, 1)`은 이 read-modify-write 세 단계를 하드웨어
  수준에서 **끊어질 수 없는 하나의 연산**으로 만들어, 다른 스레드가 중간에 끼어들지 못하게 합니다.
  대가는 있습니다 — 같은 주소를 두고 경쟁하는(contending) 스레드가 많을수록 atomic 연산들이
  **직렬화**되어 기다리는 시간이 늘어납니다. 그래서 "정확성은 atomic이 보장하지만, 성능은
  경쟁을 줄여야(예: 공유 메모리로 먼저 지역적으로 집계) 얻을 수 있다"는 것이 09의 핵심 교훈입니다.
- **`syncthreads()`의 역할**: 이건 atomic과 달리 "값을 안전하게 바꾸는" 것이 아니라, 블록 내
  모든 스레드가 **같은 지점에 도달할 때까지 서로 기다리게** 만드는 장벽(barrier)입니다. 전형적인
  쓰임은 "공유 메모리에 데이터를 쓴다 → `syncthreads()` → 그 데이터를 (다른 스레드가 쓴 값까지)
  읽는다"의 순서를 강제하는 것입니다. 이 호출이 없으면, 어떤 스레드는 아직 이웃이 쓰지 않은
  "쓰레기 값"을 읽어버릴 수 있습니다.
- **09에서의 3단계 개선**: (1) 전역 배열에 레이스가 있는 순진한 버전 → (2) 전역 배열에 직접
  atomic을 적용해 정확성은 확보하지만 경쟁이 심해 느린 버전 → (3) 블록별 **공유 메모리**에
  먼저 지역적으로 atomic 집계한 뒤, 블록이 끝날 때 전역 배열에 한 번만 합치는(더 적은 경쟁의)
  버전 — 이 진행 순서 자체가 "정확성 먼저, 그다음 성능"이라는 최적화의 기본 태도를 보여줍니다.

<a id="7"></a>
## 7. occupancy(점유율)

**occupancy** = SM에 동시에 올라간 warp 비율.
- 높을수록 지연을 잘 숨김(무조건 빠른 건 아님)
- **블록당 스레드 수·레지스터·공유메모리** 사용량이 좌우
- 그래서 `threads_per_block`·`items_per_thread` **튜닝**이 중요(08·11에서 스윕).

### 조금 더 구체적으로

**정의를 수식으로**: `occupancy = (SM에 실제로 상주하는 활성 warp 수) / (SM이 지원하는 최대 warp 수)`.
예를 들어 한 SM이 최대 64개의 warp를 동시에 품을 수 있는데 실제로는 32개만 상주한다면
occupancy는 50%입니다.

**무엇이 이 값을 제한하는가**: 아래 세 자원 중 **가장 빡빡한 것**이 "SM에 동시에 올릴 수 있는
블록(따라서 warp) 수"의 상한을 정합니다.
1. **블록당 스레드 수**: 블록 크기가 크면 SM에 동시에 올릴 수 있는 블록 수가 자연히 줄어듭니다.
2. **스레드당 레지스터 사용량**: SM의 레지스터 총량은 고정되어 있으므로, 커널 하나가 스레드마다
   레지스터를 많이 쓸수록(복잡한 계산, 많은 지역 변수) 동시에 올릴 수 있는 스레드 수가 줄어듭니다.
3. **블록당 공유 메모리 사용량**: 4절에서 본 대로 SM의 공유 메모리 총량도 고정이라, 블록이
   공유 메모리를 많이 요청할수록 동시에 상주 가능한 블록 수가 줄어듭니다.

**"100% occupancy가 항상 최선은 아니다"**: latency hiding(3절)은 "대기 중에 교대할 warp가
충분히 있는가"가 핵심이지, 무조건 최댓값을 채워야 하는 것은 아닙니다. 이미 지연을 가릴 만큼
(예: 50~70% 수준) warp가 있다면 그 이상으로 occupancy를 억지로 높이는 것(예: 블록 크기를
과도하게 줄여 레지스터를 아끼는 것)이 오히려 스레드당 일감을 늘리는 다른 최적화(레지스터
재사용, 명령어 수준 병렬성)를 희생시켜 손해가 될 수 있습니다. 즉 occupancy는 "성능을 보장하는
지표"가 아니라 "성능을 진단하는 실마리"에 가깝습니다.

> `08`·`11`에서는 `threads_per_block`(그리고 스레드당 처리 원소 수, `items_per_thread`)을
> 64·128·256·512·1024 등으로 바꿔가며 실제 처리량을 스윕(sweep)해, "이론적 occupancy"와
> "실측 성능"이 항상 정비례하지는 않는다는 것을 직접 확인합니다.

<a id="8"></a>
## 8. 개념 → 노트북 매핑

| 개념 | 07 CuPy | 08 Numba copy | 09 Numba hist | 10 cccl | 11 RawKernel |
|------|--------|----------------|----------------|---------|--------------|
| 인덱싱 | 자동 | **직접** | 직접 | 추상화 | **직접(CUDA C)** |
| coalescing | 자동 | **측정** | 적용 | 내부 | 적용 |
| 공유메모리 | — | — | **사용** | 내부 | **사용(타일링/리덕션)** |
| atomic/동기화 | — | — | **사용** | 내부 | 사용 |
| occupancy 튜닝 | — | **스윕** | 적용 | 자동 | **스윕** |
| 난이도 | 가장 쉬움 | 중간 | 높음 | 낮음 | 높음(CUDA C) |

**07** 개념은 CuPy가 대신 처리(쉬움) · **08·09** Numba로 직접 · **10** 검증 알고리즘으로 추상화 · **11** CUDA C로 직접+개념 적용 기술.

### 이 표를 어떻게 쓸까

이 표는 Day 2 전체의 **조감도**입니다. 지금 1~7절에서 정의한 다섯 개념(인덱싱·coalescing·
공유메모리·atomic/동기화·occupancy)이 이후 노트북에서 각각 "얼마나 직접 다뤄지는가"를 한눈에
보여줍니다. 표를 읽는 요령은 다음과 같습니다.

- **가로로 읽으면**: 하나의 개념(예: 공유메모리)이 난이도에 따라 어떻게 점진적으로 노출되는지
  보입니다 — `07`에서는 존재조차 안 보이다가(`—`), `09`에서 처음 직접 다루고, `11`에서
  타일링·리덕션이라는 더 정교한 용법으로 재등장합니다.
- **세로로 읽으면**: 하나의 노트북(예: `08`)이 어떤 개념들에 집중하는지 보입니다 — `08`은
  인덱싱·coalescing·occupancy 튜닝에 집중하고 공유메모리·atomic은 다루지 않습니다(그건 `09`의 몫).
- **난이도 행**이 곡선을 이룹니다: `07`(가장 쉬움, CuPy가 다 해줌) → `08`(중간, 인덱싱은 직접
  하지만 개념 하나씩) → `09`(높음, 공유메모리+atomic까지 결합) → `10`(다시 낮음, 검증된
  알고리즘 라이브러리로 추상화) → `11`(높음, CUDA C로 모든 개념을 직접 조합).

> 실전 팁: 08~11을 진행하다가 "지금 이게 왜 이렇게 동작하지?"라는 의문이 들면, 이 표로 돌아와
> 지금 보고 있는 개념이 몇 절에서 정의됐는지 먼저 확인하세요. 이 표는 이 노트북에서 한 번 보고
> 끝내는 것이 아니라, Day 2 내내 다시 펼쳐볼 **참조 지도**로 의도되었습니다.

<a id="9"></a>
## 9. ElementwiseKernel

**개념 적용**: '각 스레드가 원소 1개 처리'(2절)를 CuPy가 자동 인덱싱·런치. 우리는 **원소별 C 식**만 작성.
`ElementwiseKernel(in_params, out_params, operation, name)`.

### 조금 더 구체적으로

`ElementwiseKernel`을 처음 호출하면 CuPy는 내부적으로 다음을 자동으로 수행합니다.

1. 사용자가 넘긴 `in_params`/`out_params`/`operation` 문자열로 **완전한 CUDA C 커널 소스**를
   조립합니다 — 2절에서 본 "전역 인덱스 계산 + 경계 체크 + 원소별 연산"이 들어간 커널을 사용자
   대신 CuPy가 써주는 셈입니다.
2. 그 소스를 nvrtc(NVIDIA 런타임 컴파일러)로 **즉석 컴파일(JIT)** 합니다 — `00`에서 본
   "`cp.sin(x)`를 처음 호출할 때 커널이 컴파일·캐시된다"는 원리와 정확히 같은 메커니즘입니다.
3. 같은 dtype/shape 조합으로 다시 호출하면 컴파일된 커널을 **캐시에서 재사용**하므로 두 번째
   호출부터는 컴파일 비용이 사라집니다.
4. 입력 배열들의 shape이 서로 다르거나 스칼라가 섞여 있어도 NumPy/CuPy의 **브로드캐스팅 규칙**을
   그대로 적용해 자동으로 맞춰줍니다.

**왜 "가장 쉬운 시작"인가**: 2절에서 배운 "각 스레드가 자신의 인덱스로 담당 원소를 고른다"는
개념 중, 인덱스를 계산하는 부분(`blockIdx`/`threadIdx`/경계 체크)을 사용자가 **전혀 신경 쓸
필요가 없습니다**. 사용자는 딱 "원소 하나에 대해 무엇을 할지"만 C 표현식으로 적으면 되고,
나머지(런치 구성, 인덱싱, 경계 처리)는 모두 CuPy가 대신합니다. 이는 Day 2에서 커널 작성 난이도가
가장 낮은 지점이며, `08`에서 이 인덱싱을 Numba로 직접 쓰기 시작하면 지금 숨겨진 것이 무엇인지
체감할 수 있습니다.

In [ ]:
clamp_scale = cp.ElementwiseKernel(
    'float32 x, float32 lo, float32 hi, float32 scale', 'float32 y',
    '''
        float v = x;
        if (v < lo) v = lo;
        if (v > hi) v = hi;
        y = v * scale;
    ''', 'clamp_scale')
x = cp.random.standard_normal(10_000_000, dtype=cp.float32)
y = clamp_scale(x, cp.float32(-1), cp.float32(1), cp.float32(0.5))
print(float(y.mean()), float(y.std()))

**연습 — LeakyReLU**: `y = x>0 ? x : a*x` 를 ElementwiseKernel로 작성·검증.

위 `clamp_scale` 예제처럼 조건문 두 개(`if (v<lo)`, `if (v>hi)`)를 쓸 수도 있지만, 여기서는
`? :` 3항 연산자로 한 줄에 표현해봅니다 — 스칼라 파라미터 `a`(음수 구간의 기울기)를 추가로
받는 점도 `clamp_scale`이 `lo`/`hi`/`scale`을 추가로 받은 것과 같은 패턴입니다.

In [ ]:
# TODO: leaky = cp.ElementwiseKernel('float32 x, float32 a','float32 y','...','leaky')
x_np = np.random.randn(2_000_000).astype(np.float32); a = np.float32(0.1)
ref = np.where(x_np>0, x_np, a*x_np)
# allclose(ref, leaky(cp.asarray(x_np), cp.float32(a)), name='leaky')

<details><summary>💡 해답 보기</summary>

```python
leaky = cp.ElementwiseKernel('float32 x, float32 a','float32 y',
                             'y = x > 0 ? x : a * x;','leaky_relu')

def allclose(a, b, name="test"):
    # If b is a CuPy array, move it to CPU for NumPy comparison
    if hasattr(b, 'get'): 
        b = b.get()
    
    assert np.allclose(a, b, atol=1e-5, rtol=1e-4), f"{name} failed!"
    print(f"{name} passed!")

allclose(ref, leaky(cp.asarray(x_np), cp.float32(a)), name='leaky')
```
</details>

<a id="10"></a>
## 10. ReductionKernel

**개념 적용**: map→reduce 패턴(많은 스레드의 부분합을 합침)을 CuPy가 내부적으로 공유메모리·트리 리덕션으로 처리.
5요소: `in, out, map_expr, reduce_expr, post, identity, name`.

### 조금 더 구체적으로

**왜 "트리" 리덕션인가**: N개의 값을 하나로 합칠 때, 순차적으로 하나씩 누적하면 N-1번의 덧셈이
필요하고 이는 병렬화할 수 없습니다(다음 값을 더하려면 이전 누적 결과가 있어야 하므로). 대신
"절반씩 짝지어 동시에 더하기"를 반복하면 매 단계 남은 값이 절반으로 줄어, 총 `log2(N)` 단계만에
끝납니다 — 예를 들어 N=1024면 순차 1023단계 대신 **10단계**로 끝납니다. 여러 스레드가 동시에
자기 짝과 더할 수 있으니 이 방식이 병렬 하드웨어에 훨씬 유리합니다.

**5요소의 의미**:
- `map_expr` — 합치기 **전에** 각 원소에 먼저 적용하는 전처리(예: `fabsf(x)`로 절댓값을 취함).
- `reduce_expr` — 두 값을 하나로 합치는 연산(`a + b`). 병렬로 어떤 순서·짝짓기로 합쳐지든 항상
  같은 결과가 나와야 하므로, 반드시 **결합법칙**(그리고 이상적으로는 교환법칙)이 성립하는 연산이어야
  합니다 — 예를 들어 뺄셈(`a - b`)을 그대로 넣으면 합치는 순서에 따라 결과가 달라져 잘못됩니다.
- `identity` — `reduce_expr`의 **항등원**. 빈 블록이나 초기값으로 쓰이며, 덧셈이면 `'0.0f'`,
  곱셈이면 `'1.0f'`처럼 "그 연산에 아무 영향을 주지 않는 값"을 지정합니다.
- `post` — 모든 리덕션이 끝난 최종 값에 한 번 더 적용하는 후처리(예: 합을 원소 수로 나눠 평균을
  만들 때 `'y = a / n'`처럼 사용).

**CuPy가 숨겨서 처리하는 것**: 내부적으로는 먼저 각 블록이 자기 몫의 원소들을 **공유 메모리**에
모아 블록 내부에서 트리 리덕션을 수행하고(4절 공유 메모리, 6절 `syncthreads`가 실제로 여기서
쓰이고 있습니다 — 사용자에게는 보이지 않을 뿐), 그다음 블록별 부분합들을 다시 한 번 리덕션해
최종 결과를 만드는 **2단계 리덕션**을 자동 생성합니다.

In [ ]:
l1 = cp.ReductionKernel('float32 x','float32 y',
    'fabsf(x)', 'a + b', 'y = a', '0.0f', 'l1_norm')
x_np = np.random.randn(2_000_000).astype(np.float32)

def allclose(a, b, rtol=1e-5, atol=1e-8, name="test"):
    # If b is a CuPy array, move it to CPU for NumPy comparison
    if hasattr(b, 'get'): 
        b = b.get()
    
    # Pass the rtol and atol arguments into NumPy's allclose
    assert np.allclose(a, b, atol=atol, rtol=rtol), f"{name} failed!"
    print(f"{name} passed!")
    
allclose(np.abs(x_np).sum().astype(np.float32), l1(cp.asarray(x_np)), rtol=1e-5, atol=1e-4, name='L1')

**연습 — 제곱합**: `map=x*x, reduce=a+b` 로 ReductionKernel 작성.

위 `l1` 예제와 구조는 동일하고 `map_expr`만 바뀝니다 — `fabsf(x)`(절댓값) 대신 `x*x`(제곱)를
넣으면 됩니다. `reduce_expr`(`a+b`)과 `identity`(`'0.0f'`)는 그대로 재사용할 수 있습니다.

In [ ]:
# TODO: ss = cp.ReductionKernel('float32 x','float32 y','x*x','a+b','y=a','0.0f','sum_sq')
x_np = np.random.randn(2_000_000).astype(np.float32)
ref = (x_np**2).sum().astype(np.float32)
# allclose(ref, ss(cp.asarray(x_np)), rtol=1e-4, atol=1e-1, name='sum_sq')

<details><summary>💡 해답 보기</summary>

```python
ss = cp.ReductionKernel('float32 x','float32 y','x*x','a+b','y=a','0.0f','sum_sq')
allclose(ref, ss(cp.asarray(x_np)), rtol=1e-4, atol=1e-1, name='sum_sq')
```
</details>

<a id="11"></a>
## 11. `@cupy.fuse` 와의 관계

- 단순 원소/리덕션 융합이면 **`@cupy.fuse`**(03)가 가장 쉽습니다. C 식 제어가 필요하면 Elementwise/Reduction,
- 복잡한 인덱싱·공유메모리·atomic이 필요하면 → **Numba CUDA(08~09)**, CUDA C 직접 작성은 → **RawKernel(11)**.
- 평소 쓰던 일반 파이썬/CuPy 코드 위에 @cp.fuse() 데코레이터 딱 한 줄만 붙여주면 CuPy가 알아서 초고속 GPU 커널로 융합(Fusion)해 주는 가장 쉬운 최적화 도구

### 조금 더 구체적으로 — 무엇이 다른가

**`fuse`의 원리**: 이미 존재하는 여러 개의 CuPy 원소별 연산(ufunc) 호출을 연달아 쓰면, 원래는
각 연산마다 커널이 하나씩 런치되고 그 사이 중간 결과가 매번 전역 메모리에 왕복합니다(써졌다가
다시 읽힘). `@cp.fuse()`는 이 연산 체인을 분석해 **하나의 커널**로 합쳐, 중간 결과를 레지스터에만
두고 전역 메모리 왕복과 커널 런치 횟수를 동시에 줄입니다 — `03_numpy_routines`에서 다룬 내용의
연장선입니다.

**`ElementwiseKernel`/`ReductionKernel`과의 차이**: `fuse`는 "이미 있는 CuPy 함수들의 **조합**"을
자동으로 최적화하는 도구인 반면, `ElementwiseKernel`/`ReductionKernel`은 사용자가 **C 표현식
자체를 직접 작성**해 CuPy에 없는 연산(3항 연산자, `raw` 이웃 인덱싱, 커스텀 수식)을 표현하는
도구입니다. 즉 "CuPy 함수 조합만으로 표현 가능한가?"가 두 접근을 가르는 기준입니다.

**선택 기준(난이도 순)**:
1. 표준 CuPy 함수들의 조합으로 표현되는가? → **`@cp.fuse`**(가장 쉬움).
2. 원소별/리덕션 연산이지만 CuPy에 없는 C 레벨 표현식(조건, 이웃 접근)이 필요한가? →
   **`ElementwiseKernel`/`ReductionKernel`**(이 노트북).
3. 공유 메모리·atomic·복잡한 인덱싱이 필요한가? → **Numba CUDA**(`08`~`09`).
4. CUDA C 코드 자체를 완전히 직접 통제해야 하는가(타일링, 정교한 최적화)? → **RawKernel**(`11`).

이 순서는 8절 매핑표의 "난이도" 행과 정확히 같은 축입니다 — 위로 갈수록 쉽고 CuPy가 많이
대신해주며, 아래로 갈수록 사용자가 더 많은 것을 직접 제어합니다.

## 🧪 추가 연습 — 커널 더 다루기

지금까지 익힌 `ElementwiseKernel`/`ReductionKernel`의 기본 5~6요소를 다양한 변형(다중 입력,
`raw` 이웃 접근, 가중 리덕션)에 적용해보며 손에 익히는 연습입니다. 정답을 보기 전에 먼저
스스로 C 표현식을 써보세요 — 09절의 `l1`, 이 절 앞의 `clamp_scale` 예제 구조를 그대로 본떠도 됩니다.

**연습 A — 2입력 ElementwiseKernel**: `z = a*x + b*y` 를 작성하세요(입력 4개).

`in_params`에 배열 두 개(`x`, `y`)와 스칼라 두 개(`a`, `b`)를 함께 나열하면 됩니다 — CuPy는
배열과 스칼라를 구분해 스칼라는 모든 스레드에 동일한 값으로, 배열은 원소별로 넘겨줍니다.

In [ ]:
# TODO: axpby = cp.ElementwiseKernel('float32 x, float32 y, float32 a, float32 b','float32 z','...','axpby')
x = cp.random.rand(1_000_000, dtype=cp.float32); y = cp.random.rand(1_000_000, dtype=cp.float32)
# allclose(2*cp.asnumpy(x)+3*cp.asnumpy(y), axpby(x,y,cp.float32(2),cp.float32(3)), name='axpby')

<details><summary>💡 해답 보기</summary>

```python
axpby = cp.ElementwiseKernel('float32 x, float32 y, float32 a, float32 b','float32 z',
                             'z = a*x + b*y;','axpby')
allclose(2*cp.asnumpy(x)+3*cp.asnumpy(y), axpby(x,y,cp.float32(2),cp.float32(3)), name='axpby')
```
</details>

**연습 B — `raw` 파라미터로 이웃 접근**: 
- ElementwiseKernel은 `raw`(임의 인덱싱)+루프 인덱스 `i`로 이웃 원소에 접근할 수 있습니다.
- 전진 차분 `y[i] = x[i+1]-x[i]`(마지막은 0)을 작성하세요.
- ElementwiseKerne에서 raw 키워드를 붙이면 CuPy가 자동으로 수행하는 1:1 인덱싱을 해제하고, C++ 스타일의 포인터 배열처럼 x[i+1]이나 x[i-1] 같은 임의의 인덱스(이웃 원소)에 자유롭게 접근할 수 있게 해줍니다. 이때 자동으로 제공되는 루프 변수 i는 현재 GPU 스레드가 처리 중인 원소의 인덱스 번호를 나타냅니다.

**왜 이게 필요한가**: 지금까지의 예제(`clamp_scale`, `axpby` 등)는 모두 "내 원소만" 보면 되는
순수 원소별 연산이었습니다. 하지만 차분(diff), 스텐실(stencil), 컨볼루션처럼 **이웃 원소**를
함께 봐야 하는 연산은 CuPy의 기본 1:1 자동 인덱싱만으로는 표현할 수 없습니다. `raw`는 이 자동
인덱싱을 해제해, 2절에서 배운 "전역 인덱스"(여기서는 `i`로 자동 제공됨) 자체를 이용해 `x[i-1]`,
`x[i+1]`처럼 자유롭게 이웃에 접근하게 해줍니다. 이런 이웃 접근·타일링 패턴은 `11`의 2D 스텐실
RawKernel에서 공유 메모리와 함께 훨씬 본격적으로 다시 등장합니다.

In [ ]:
# TODO: fwd = cp.ElementwiseKernel('raw float32 x, int32 n','float32 y',
#           'y = (i < n-1) ? x[i+1]-x[i] : 0.0f;', 'fwd_diff')
x = cp.arange(10, dtype=cp.float32); y = cp.empty(10, dtype=cp.float32)
# fwd(x, np.int32(x.size), y); print(cp.asnumpy(y))   # 1,1,...,0

<details><summary>💡 해답 보기</summary>

```python
fwd = cp.ElementwiseKernel('raw float32 x, int32 n','float32 y',
                           'y = (i < n-1) ? x[i+1]-x[i] : 0.0f;', 'fwd_diff')
x = cp.arange(10, dtype=cp.float32); y = cp.empty(10, dtype=cp.float32)
fwd(x, np.int32(x.size), y); print(cp.asnumpy(y))
# raw 입력은 브로드캐스팅으로 크기를 못 정하므로 출력 배열(y)로 루프 크기를 지정합니다.
```
</details>

**연습 C — 가중합 ReductionKernel**: `sum(x*w)` 를 ReductionKernel로 작성(입력 2개).

10절의 `l1`과 뼈대는 같습니다 — 이번엔 입력이 `x` 하나가 아니라 `x, w` 두 개이므로 `map_expr`에서
두 입력을 함께 사용해(`x*w`) 곱한 뒤, `reduce_expr`(`a+b`)로 합칩니다. 두 개 이상의 배열을
동시에 리듀스하는 이 패턴은 내적(dot product)·가중평균 계산의 기본형입니다.

In [ ]:
# TODO: wsum = cp.ReductionKernel('float32 x, float32 w','float32 y','x*w','a+b','y=a','0.0f','wsum')
x = np.random.rand(1_000_000).astype(np.float32); w = np.random.rand(1_000_000).astype(np.float32)
# allclose((x*w).sum().astype('f4'), wsum(cp.asarray(x), cp.asarray(w)), rtol=1e-3, atol=1e-1, name='wsum')

<details><summary>💡 해답 보기</summary>

```python
wsum = cp.ReductionKernel('float32 x, float32 w','float32 y','x*w','a+b','y=a','0.0f','wsum')
allclose((x*w).sum().astype('f4'), wsum(cp.asarray(x), cp.asarray(w)), rtol=1e-3, atol=1e-1, name='wsum')
```
</details>

<a id="12"></a>
## 12. 체크포인트

- [ ] 커널·thread/block/grid·SIMT/warp를 설명할 수 있다
- [ ] 메모리 계층·coalescing·atomic/동기화·occupancy 개념을 안다
- [ ] 각 개념이 07~11에서 어떻게 구현되는지 매핑을 이해했다
- [ ] ElementwiseKernel·ReductionKernel로 커널을 작성했다

다음: **`08_numba_copy`** — Numba CUDA로 스레드 인덱싱과 **coalescing**을 직접 구현·측정합니다.